# Distinguish process variation from uncertainty

Level: 1. Prerequisites: Level 1 of learning. Reading time: 18 minutes.

## Objectives

Read the committed uncertainty record, identify its uncertain ore inputs and output quantiles, and explain why an interval around one simulated state is not a plant confidence interval.

## Setup

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0
```

The OreFlow pipeline is imported from `data-pipeline/`; this notebook never installs packages.

Configure imports and record versions. This makes the method run reproducible.

In [1]:
# Import analysis tools and record their exact versions.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the local pipeline from the repository-relative layout.
repo = Path('../../../').resolve()
sys.path.insert(0, str(repo / 'data-pipeline'))
for package in ('numpy', 'scipy', 'pandas', 'matplotlib', 'nbclient', 'nbformat', 'ipykernel'):
    print(package, importlib.metadata.version(package))

numpy 2.2.6
scipy 1.17.1
pandas 2.2.3
matplotlib 3.11.2
nbclient 0.11.0
nbformat 5.11.1
ipykernel 7.4.0


## Data

Owner: OreFlow authors; `data/derived/benchmark.json` and `data/derived/studies.json`, MIT project licence. The ranges are authored uncertainty assumptions applied to authored process cases; labels are simulated. Repository: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Method

OreFlow varies work index, head grade, liberation size and floatability within declared multiplicative ranges, then re-solves the same steady-state circuit. A seeded Latin hypercube produces reproducible samples. Quantiles describe this model and these ranges; they are not estimates of field measurement error or plant-to-plant variability.

Load the uncertainty quantiles and the repeated-seed study for the same authored case.

In [2]:
# Load the uncertainty quantiles and the repeated-seed study for the same authored case.
repo = Path('../../../').resolve()
benchmark = json.loads((repo / 'data/derived/benchmark.json').read_text(encoding='utf-8'))
studies = json.loads((repo / 'data/derived/studies.json').read_text(encoding='utf-8'))
record = benchmark['uncertainty']['copper_porphyry_soft']
seed_study = studies['cases']['copper_porphyry_soft']['seed_study']
print('design:', seed_study['design'], '| samples:', seed_study['samples'], '| seeds:', len(seed_study['seeds']))
display(pd.DataFrame([{'output': name, **summary} for name, summary in record.items() if isinstance(summary, dict) and 'p50' in summary]).set_index('output'))

design: Latin hypercube | samples: 128 | seeds: 8


,p05,p50,p95
output,,,
recovery_pct,89.410940,91.897183,93.795676
concentrate_grade,24.868232,26.441278,27.709650


Compare the input that dominates recovery spread with the P05-P95 output range. These are properties of the declared simulation envelope.

In [3]:
# Compare the input that dominates recovery spread with the P05-P95 output range.
print('dominant input by output:', record['dominant_input'])
print('all-constraint probability:', record['probabilities']['all_constraints'])
seed_rows = pd.DataFrame([{'seed': item['seed'], 'recovery P05': item['recovery_pct']['p05'], 'recovery P50': item['recovery_pct']['p50'], 'recovery P95': item['recovery_pct']['p95'], 'all constraints': item['all_constraints']} for item in seed_study['per_seed']])
display(seed_rows)

dominant input by output: {'recovery_pct': 'floatability', 'concentrate_grade': 'liberation_size', 'specific_energy_grinding_kwh_t': 'work_index', 'recovered_primary_tph': 'head_grade'}
all-constraint probability: 0.8203125


,seed,recovery P05,recovery P50,recovery P95,all constraints
0,101,89.216297,91.996411,94.025237,0.820312
1,202,89.018685,91.983648,94.034888,0.820312
2,303,89.042295,91.985040,93.914918,0.812500
3,404,89.367639,91.975672,93.875840,0.812500
4,505,89.079242,92.035955,93.745444,0.820312
5,606,89.087326,91.944077,93.799518,0.812500
6,707,89.374171,92.092473,94.007753,0.812500
7,808,89.294996,91.917077,93.948797,0.812500


## Results

The quantiles summarize model output conditional on the authored half-widths and the chosen case. The seed study shows how estimates shift with a different design. Neither describes observational uncertainty in a real plant unless independent measurements and a justified uncertainty model support that interpretation.

## Summary

You separated a seeded model uncertainty study from measurement uncertainty. State the sampling design, input ranges and data provenance whenever quoting these results.

## References

- https://doi.org/10.1080/00401706.1979.10489755